In [0]:
from pyspark.sql.types import *
import pyspark.sql.functions as F

In [0]:
def ingest_into_delta_lake(dim: str, schema: StructType):
    raw_data_path = f'/Volumes/ecommerce/source_data/raw/{dim}/*.csv'

    df = spark.read.csv(raw_data_path, header=True, schema=schema)

    # Add metadata columns
    df = df.withColumn('_souce_file', F.col('_metadata.file_path'))\
        .withColumn('ingested_at',F.current_timestamp())
    
    df.write.mode('overwrite')\
        .format('delta')\
        .option('mergeSchema', 'true')\
        .saveAsTable(f'ecommerce.bronze.bronze_{dim}')

In [0]:
brands_schema = StructType([
    StructField('brand_code', StringType(), False),
    StructField('brand_name', StringType(), True),
    StructField('category_code', StringType(), True)
])

category_schema = StructType([
    StructField('category_code', StringType(), False),
    StructField('category_name', StringType(), True)
])

products_schema = StructType([
    StructField('product_id', StringType(), False),
    StructField('sku', StringType(), True),
    StructField('category_code', StringType(), True),
    StructField('brand_code', StringType(), True),
    StructField('color', StringType(), True),
    StructField('size', StringType(), True),
    StructField('material', StringType(), True),
    StructField('weight_grams', StringType(), True),
    StructField('lenght_cm', StringType(), True),
    StructField('widht_cm', FloatType(), True),
    StructField('height_cm', FloatType(), True),
    StructField('rating_count', IntegerType(), True)
])

customers_schema = StructType([
    StructField('customer_id', StringType(), False),
    StructField('phone', StringType(), True),
    StructField('country_code', StringType(), True),
    StructField('country', StringType(), True),
    StructField('state', StringType(), True)
])

date_schema = StructType([
    StructField('date', StringType(), True),
    StructField('year', IntegerType(), True),
    StructField('day_name', StringType(), True),
    StructField('quarter', IntegerType(), True),
    StructField('week_of_year', IntegerType(), True)
])

In [0]:
ingest_into_delta_lake('brands', brands_schema)
ingest_into_delta_lake('category', category_schema)
ingest_into_delta_lake('products', products_schema)
ingest_into_delta_lake('customers', customers_schema)
ingest_into_delta_lake('date', date_schema)